In [1]:
import torch
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import numpy as np
import cv2
import os
from torch import nn

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import matplotlib.pyplot as plt



In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)



def preprocess_image(image, train=True):

    img = np.array(image)

    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

    l, a, b = cv2.split(img)

    clahe = cv2.createCLAHE(clipLimit=1.5)
    cl = clahe.apply(l)

    limg = cv2.merge((cl, a, b))
    img = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

    image = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    if train:

        transform = transforms.Compose([
            transforms.Resize((256,256)),
            transforms.RandomResizedCrop(224),
            transforms.RandomHorizontalFlip(),
             transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485,0.456,0.406],
                std=[0.229,0.224,0.225]
            )

        ])

    else:

        transform = transforms.Compose([
            transforms.Resize((224,224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485,0.456,0.406],
                std=[0.229,0.224,0.225]
            )
        ])

    image_tensor = transform(image)

    return image_tensor



cuda


In [3]:
def collect_images(dataset_path):

    classes = sorted(os.listdir(dataset_path))
    class_to_idx = {c:i for i,c in enumerate(classes)}

    images = []

    valid_ext = (".jpg", ".jpeg", ".png")

    for c in classes:

        class_folder = os.path.join(dataset_path, c)

        for img in os.listdir(class_folder):

            if not img.lower().endswith(valid_ext):
                continue

            path = os.path.join(class_folder, img)

            images.append((path, class_to_idx[c]))

    return images, classes,class_to_idx

def extract_leaf_mask(image):
    """
    Isolates the leaf from the background.
    Returns the binary mask of the leaf and the total leaf area.
    """
    hsv_image=cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    LOWER_LEAF = np.array([15, 40, 40])
    UPPER_LEAF = np.array([95, 255, 255])

    # Spot detection thresholds


    KERNEL = np.ones((5, 5), np.uint8)
    # 1. Create a mask based on color range
    mask = cv2.inRange(hsv_image, LOWER_LEAF, UPPER_LEAF)

    # 2. Clean up noise using Morphological Closing
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, KERNEL)

    # 3. Find contours to identify distinct objects
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return None, 0

    # 4. Keep only the largest contour (assuming it is the leaf)
    largest = max(contours, key=cv2.contourArea)

    # 5. Draw a clean mask of just the largest contour
    final_leaf_mask = np.zeros_like(mask)
    cv2.drawContours(final_leaf_mask, [largest], -1, 255, -1)

    leaf_area = cv2.countNonZero(final_leaf_mask)
    leaf = cv2.bitwise_and(image, image, mask=final_leaf_mask)
    return leaf, final_leaf_mask

In [4]:
class LeafDataset(Dataset):

    def __init__(self, data, train=True):

        self.data = data
        self.train = train

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        img_path, label = self.data[idx]

        with Image.open(img_path) as img:
            image = img.convert("RGB")

        image = preprocess_image(image, train=self.train)

        return image, label

In [5]:
dataset_path="Datasets/dataset_yolo2"

images, classes, class_to_idx = collect_images(dataset_path)
train_path = "Datasets/dataset_yolo2/images/train"
val_path   = "Datasets/dataset_yolo2/images/val"
test_path  = "Datasets/dataset_yolo2/images/test"

train_data, classes, class_to_idx = collect_images(train_path)
val_data, _, _ = collect_images(val_path)
test_data, _, _ = collect_images(test_path)
labels = [label for _, label in train_data]

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

In [6]:
train_dataset = LeafDataset(train_data, train=True)
val_dataset   = LeafDataset(val_data, train=False)
test_dataset  = LeafDataset(test_data, train=False)

In [8]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [9]:
class_num=len(classes)
model = models.resnet50(weights="DEFAULT", progress=True)
model.fc = nn.Linear(model.fc.in_features, class_num)

model = model.to(device)

# ---------- Freeze backbone ----------
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

model=model.to(device)
loss_fun=nn.CrossEntropyLoss(weight=class_weights)
optimizer=torch.optim.Adam(model.fc.parameters(),lr=3e-4)
epochs=20
patience = 5
best_val_loss = float("inf")
counter = 0


In [34]:

freeze_epochs = 3
for epoch in range(epochs):
    if epoch == freeze_epochs:
        print("Unfreezing backbone...")

        for param in model.parameters():
            param.requires_grad = True
        optimizer=torch.optim.Adam(model.parameters(),lr=1e-4)

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for images, labels in loop:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = loss_fun(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        loop.set_postfix(
            loss=running_loss/len(train_loader),
            acc=100*correct/total
        )

    model.eval()

    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = loss_fun(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = val_loss / len(val_loader)
    val_acc = 100 * correct / total

    print(f"\nValidation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.2f}%")

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        counter = 0

        torch.save(model.state_dict(), "best_model_resnet.pth")

        print("Validation improved, model saved!")

    else:

        counter += 1
        print(f"No improvement for {counter} epochs")

        if counter >= patience:
            print("Early stopping triggered!")
            break

Epoch 1/20: 100%|██████████| 855/855 [02:24<00:00,  5.93it/s, acc=62.1, loss=1.6]  



Validation Loss: 1.1728
Validation Accuracy: 69.20%
Validation improved, model saved!


Epoch 2/20: 100%|██████████| 855/855 [01:59<00:00,  7.15it/s, acc=74.2, loss=1.01] 



Validation Loss: 0.9027
Validation Accuracy: 75.21%
Validation improved, model saved!


Epoch 3/20: 100%|██████████| 855/855 [01:59<00:00,  7.14it/s, acc=77.5, loss=0.839]


...


Validation Loss: 0.0033
Validation Accuracy: 99.95%
Validation improved, model saved!


Epoch 19/20: 100%|██████████| 855/855 [03:38<00:00,  3.91it/s, acc=98.3, loss=0.0515] 



Validation Loss: 0.0121
Validation Accuracy: 99.55%
No improvement for 1 epochs


Epoch 20/20: 100%|██████████| 855/855 [03:38<00:00,  3.91it/s, acc=98.4, loss=0.0504] 



Validation Loss: 0.0092
Validation Accuracy: 99.75%
No improvement for 2 epochs


In [10]:

model.load_state_dict(torch.load("best_model_resnet.pth",map_location=device,weights_only=True))

<All keys matched successfully>

In [11]:
from sklearn.metrics import classification_report
model.eval()

test_loss = 0
correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():

    loop = tqdm(test_loader, desc="Testing")

    for images, labels in loop:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = loss_fun(outputs, labels)

        test_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_loss /= len(test_loader)
test_acc = 100 * correct / total

print("\nTest Results")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=classes))

Testing: 100%|██████████| 176/176 [00:25<00:00,  6.86it/s]


Test Results
Test Loss: 0.0047
Test Accuracy: 99.88%

Classification Report:
                                             precision    recall  f1-score   support

              Pepper__bell___Bacterial_spot       1.00      1.00      1.00       151
                      Potato___Early_blight       1.00      1.00      1.00       274
                       Potato___Late_blight       1.00      1.00      1.00       283
                      Tomato_Bacterial_spot       1.00      1.00      1.00       581
                        Tomato_Early_blight       0.99      1.00      1.00       284
                         Tomato_Late_blight       1.00      1.00      1.00       531
                           Tomato_Leaf_Mold       1.00      1.00      1.00       272
                  Tomato_Septoria_leaf_spot       1.00      1.00      1.00       495
Tomato_Spider_mites_Two_spotted_spider_mite       1.00      1.00      1.00       464
                        Tomato__Target_Spot       1.00      1.00      1

In [ ]:
target_layers = [model.layer3[-1]]
gradcam_dirs = [
    "Datasets/dataset_yolo2/images/train",
    "Datasets/dataset_yolo2/images/val"
]
output_dir = "gradcam_results"
os.makedirs(output_dir, exist_ok=True)
cam = GradCAMPlusPlus(model=model, target_layers=target_layers)

labels_dir = "yolo_labels"
os.makedirs(labels_dir, exist_ok=True)


In [ ]:
# for dataset_dir in gradcam_dirs:
#
#     split = os.path.basename(dataset_dir)   # train or val
#
#     for class_name in os.listdir(dataset_dir):
#
#         class_folder = os.path.join(dataset_dir, class_name)
#
#         if not os.path.isdir(class_folder):
#             continue
#
#         for img_name in tqdm(os.listdir(class_folder), desc=f"{split} - {class_name}"):
#
#             if not img_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
#                 continue
#
#             img_path = os.path.join(class_folder, img_name)
#
#             try:
#                 image = Image.open(img_path).convert("RGB")
#
#                 original_img = np.array(image)
#                 original_img = cv2.resize(original_img, (224, 224))
#
#                 # ---------- Leaf Segmentation ----------
#                 leaf, leaf_mask = extract_leaf_mask(original_img)
#
#                 if leaf is None:
#                     continue
#
#                 # Image used for GradCAM visualization
#                 rgb_img = original_img.astype(np.float32) / 255.0
#
#                 # Tensor for model
#                 image_tensor = preprocess_image(Image.fromarray(original_img), train=False).unsqueeze(0).to(device)
#
#                 # ---------- Prediction ----------
#                 outputs = model(image_tensor)
#                 class_id = outputs.argmax(dim=1).item()
#                 targets = [ClassifierOutputTarget(class_id)]
#
#                 is_healthy = "healthy" in class_name.lower()
#                 # ---------- GradCAM ----------
#                 if is_healthy:
#                     mask = leaf_mask
#                 else:
#                     grayscale_cam = cam(input_tensor=image_tensor, targets=targets)
#
#                     if grayscale_cam is None:
#                         continue
#                     try:
#                         grayscale_cam = cam(input_tensor=image_tensor, targets=targets)
#                     except Exception as cam_error:
#                         print("GradCAM failed:", img_name, cam_error)
#                         continue
#
#                     if grayscale_cam is None:
#                         continue
#
#                     # restrict GradCAM to leaf area
#                     grayscale_cam = grayscale_cam * (leaf_mask / 255.0)
#
#                     cam_min = grayscale_cam.min()
#                     cam_max = grayscale_cam.max()
#
#                     if cam_max - cam_min == 0:
#                         continue
#
#                     heatmap_norm = (grayscale_cam - cam_min) / (cam_max - cam_min)
#
#                     threshold = np.percentile(heatmap_norm, 75)
#                     mask = (heatmap_norm >= threshold).astype("uint8") * 255
#
#                     # ---------- Clean Mask ----------
#                     kernel = np.ones((3, 3), np.uint8)
#                     mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
#                     mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
#
#                     img_h, img_w = mask.shape
#
#                     # ---------- Save mask for UNet ----------
#                     mask_dir = "unet_masks"
#                     os.makedirs(mask_dir, exist_ok=True)
#
#                     mask_path = os.path.join(mask_dir, os.path.splitext(img_name)[0] + ".png")
#                     cv2.imwrite(mask_path, mask)
#
#                     mask_color = original_img.copy()
#                     mask_color[mask == 255] = [255, 0, 0]
#
#                     # ---------- Contours ----------
#                     contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
#                     contours = sorted(contours, key=cv2.contourArea, reverse=True)[:100]
#
#                     # ---------- YOLO labels ----------
#                     labels_split_dir = os.path.join("Datasets/dataset_yolo2/labels", split)
#                     os.makedirs(labels_split_dir, exist_ok=True)
#
#                     label_path = os.path.join(labels_split_dir, os.path.splitext(img_name)[0] + ".txt")
#
#                     visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
#
#                     min_area = 0.001 * img_h * img_w
#
#                     with open(label_path, "w") as f:
#
#                         for contour in contours:
#
#                             area = cv2.contourArea(contour)
#
#                             if area < min_area:
#                                 continue
#
#                             x, y, w, h = cv2.boundingRect(contour)
#
#                             if w < 5 or h < 5:
#                                 continue
#
#                             x_center = (x + w / 2) / img_w
#                             y_center = (y + h / 2) / img_h
#                             width = w / img_w
#                             height = h / img_h
#
#                             f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")
#
#                     # ---------- Visualization ----------
#                     combined = np.concatenate([
#                         original_img,
#                         visualization,
#                         mask_color
#                     ], axis=1)
#
#                     os.makedirs(output_dir, exist_ok=True)
#
#                     save_path = os.path.join(output_dir, f"{split}_{class_name}_{img_name}")
#
#                     cv2.imwrite(save_path, cv2.cvtColor(combined, cv2.COLOR_RGB2BGR))
#
#             except Exception as e:
#                 print("Error:", img_name, e)

In [ ]:

# --- Move directory creation OUTSIDE the loops ---
unet_mask_dir = "unet_masks"
yolo_base_dir = "Datasets/dataset_yolo2"
os.makedirs(unet_mask_dir, exist_ok=True)
# Assuming output_dir is defined elsewhere, e.g., output_dir = "visualizations"
os.makedirs(output_dir, exist_ok=True)

for dataset_dir in gradcam_dirs:
    split = os.path.basename(dataset_dir)   # train or val

    yolo_labels_dir = os.path.join(yolo_base_dir, "labels", split)
    yolo_images_dir = os.path.join(yolo_base_dir, "images", split)
    os.makedirs(yolo_labels_dir, exist_ok=True)
    os.makedirs(yolo_images_dir, exist_ok=True)

    for class_name in os.listdir(dataset_dir):
        class_folder = os.path.join(dataset_dir, class_name)

        if not os.path.isdir(class_folder):
            continue

        for img_name in tqdm(os.listdir(class_folder), desc=f"{split} - {class_name}"):
            if not img_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
                continue

            img_path = os.path.join(class_folder, img_name)
            base_name = os.path.splitext(img_name)[0]

            try:
                image = Image.open(img_path).convert("RGB")
                original_img = cv2.resize(np.array(image), (224, 224))

                # ---------- Leaf Segmentation ----------
                leaf, leaf_mask = extract_leaf_mask(original_img)
                if leaf is None:
                    continue

                rgb_img = original_img.astype(np.float32) / 255.0
                image_tensor = preprocess_image(Image.fromarray(original_img), train=False).unsqueeze(0).to(device)

                # ---------- Prediction ----------
                outputs = model(image_tensor)
                class_id = outputs.argmax(dim=1).item()
                targets = [ClassifierOutputTarget(class_id)]

                is_healthy = "healthy" in class_name.lower()

                # Copy image to YOLO dataset
                yolo_img_path = os.path.join(yolo_images_dir, f"{class_name}_{img_name}")
                cv2.imwrite(yolo_img_path, cv2.cvtColor(original_img, cv2.COLOR_RGB2BGR))

                # Open YOLO label file (will remain empty if healthy)
                label_path = os.path.join(yolo_labels_dir, f"{class_name}_{base_name}.txt")

                # ---------- Mask Generation ----------
                if is_healthy:
                    mask = leaf_mask
                    visualization = rgb_img

                    img_h, img_w = mask.shape
                    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

                    with open(label_path, "w") as f:
                        if contours:
                            largest = max(contours, key=cv2.contourArea)
                            x, y, w, h = cv2.boundingRect(largest)

                            x_center = (x + w / 2) / img_w
                            y_center = (y + h / 2) / img_h
                            width = w / img_w
                            height = h / img_h

                            f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")
                else:
                    try:
                        grayscale_cam = cam(input_tensor=image_tensor, targets=targets)
                    except Exception as cam_error:
                        print(f"GradCAM failed for {img_name}: {cam_error}")
                        continue

                    if grayscale_cam is None:
                        continue

                    # Extract from batch if necessary (shape check)
                    if len(grayscale_cam.shape) == 3:
                        grayscale_cam = grayscale_cam[0]

                    # Restrict GradCAM to leaf area
                    leaf_mask_float = leaf_mask.astype(np.float32) / 255.0
                    grayscale_cam = grayscale_cam * leaf_mask_float

                    cam_min, cam_max = grayscale_cam.min(), grayscale_cam.max()
                    if cam_max - cam_min == 0:
                        continue

                    heatmap_norm = (grayscale_cam - cam_min) / (cam_max - cam_min)
                    threshold = np.percentile(heatmap_norm, 75)
                    mask = (heatmap_norm >= threshold).astype("uint8") * 255

                    # Clean Mask
                    kernel = np.ones((3, 3), np.uint8)
                    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
                    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

                    visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

                    # ---------- YOLO Bounding Boxes ----------
                    img_h, img_w = mask.shape
                    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    contours = sorted(contours, key=cv2.contourArea, reverse=True)[:100]
                    min_area = 0.001 * img_h * img_w

                    with open(label_path, "w") as f:
                        for contour in contours:
                            area = cv2.contourArea(contour)
                            if area < min_area: continue

                            x, y, w, h = cv2.boundingRect(contour)
                            if w < 5 or h < 5: continue

                            x_center = (x + w / 2) / img_w
                            y_center = (y + h / 2) / img_h
                            width = w / img_w
                            height = h / img_h

                            f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")

                # ---------- Save UNet Mask (For both healthy & diseased) ----------
                mask_path = os.path.join(unet_mask_dir, f"{class_name}_{base_name}.png")
                cv2.imwrite(mask_path, mask)

                # ---------- Final Visualization ----------
                mask_color = original_img.copy()
                mask_color[mask == 255] = [255, 0, 0]

                # Convert visualization to uint8 if it isn't already
                if visualization.dtype != np.uint8:
                    visualization = (visualization * 255).astype(np.uint8)

                combined = np.concatenate([original_img, visualization, mask_color], axis=1)
                save_path = os.path.join(output_dir, f"{split}_{class_name}_{img_name}")
                cv2.imwrite(save_path, cv2.cvtColor(combined, cv2.COLOR_RGB2BGR))

            except Exception as e:
                print(f"Error processing {img_name}: {e}")